In [ ]:
%%capture
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os, io, json, pickle, shutil, math
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, List, Tuple
from datetime import datetime
from collections import Counter
from copy import deepcopy

import numpy as np
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.amp import autocast, GradScaler

import timm
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
import torchvision.transforms as transforms

# ─────────────────────────────────────────────────────────────────────────────
# REPRODUCIBILITY
# ─────────────────────────────────────────────────────────────────────────────
torch.manual_seed(42); np.random.seed(42)

device        = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_gpus      = torch.cuda.device_count()
use_multi_gpu = num_gpus > 1

print(f"Device      : {device}")
print(f"GPUs        : {num_gpus}")
if torch.cuda.is_available():
    for i in range(num_gpus):
        print(f"  GPU {i}     : {torch.cuda.get_device_name(i)}")

# ─────────────────────────────────────────────────────────────────────────────
# CONFIG
# Speed fix: Xception @ 160×160 (was 224), batch=128 (was 64)
# Compute scales as (160/224)^2 ≈ 0.51 + 2× larger batch → ~4× faster
# Was: ~80 min/epoch. Now: ~20 min/epoch → 10 epochs ≈ 3.5 hrs on 2×T4 ✓
# ─────────────────────────────────────────────────────────────────────────────
@dataclass
class Config:
    # ── Architecture ─────────────────────────────────────────────────────────
    model_name:      str   = "xception"
    img_size:        int   = 160                  # 224→160 halves compute; Xception works fine at 160
    in_channels:     int   = 3
    backbone_out:    int   = 2048                 # Xception feature dim
    num_devices:     int   = 7                    # uav1…uav7   (PRIMARY task)
    num_distances:   int   = 4                    # 6ft/9ft/12ft/15ft (SECONDARY task)
    # ── Batch / gradient ─────────────────────────────────────────────────────
    batch_size:      int   = 128                  # fits 2×T4 with EfficientNet-B0
    grad_accum_steps: int  = 1                    # no accum needed at batch=128
    # ── Optimiser ────────────────────────────────────────────────────────────
    head_lr:         float = 1e-3
    backbone_lr:     float = 1e-4
    weight_decay:    float = 0.01
    # ── Schedule ─────────────────────────────────────────────────────────────
    epochs:          int   = 12
    patience:        int   = 5
    freeze_epochs:   int   = 2                    # shorter freeze → more fine-tune time
    grad_clip_norm:  float = 1.0
    label_smoothing: float = 0.10
    mixup_alpha:     float = 0.3
    dropout:         float = 0.4
    # ── DataLoader ───────────────────────────────────────────────────────────
    num_workers:     int   = 4
    prefetch_factor: int   = 2
    preload_images:  bool  = False
    # ── Paths ────────────────────────────────────────────────────────────────
    data_root:  str = "/dev/shm/genesys_data"
    snr_levels: Tuple[str, ...] = (
        "clean", "snr_20dB", "snr_15dB", "snr_10dB", "snr_5dB", "snr_0dB"
    )

cfg = Config()

# Only 2 tasks — SNR level is NOT a task (it is a data-partitioning axis)
TASKS        = ["device_id", "distance"]
DEVICE_MAP   = {f"uav{i}": i - 1 for i in range(1, 8)}
DISTANCE_MAP = {"6ft": 0, "9ft": 1, "12ft": 2, "15ft": 3}

print(f"\nModel       : {cfg.model_name}")
print(f"Image size  : {cfg.img_size}×{cfg.img_size}")
print(f"Batch size  : {cfg.batch_size}  (grad_accum ×{cfg.grad_accum_steps})")
print(f"Epochs      : {cfg.epochs}  Patience: {cfg.patience}")
print(f"Freeze phase: {cfg.freeze_epochs} epochs")
print(f"Tasks       : {TASKS}  ({cfg.num_devices} + {cfg.num_distances} classes)")
print(f"Est. time   : ~{cfg.epochs * 8} min total (vs ~{cfg.epochs * 80} min before)")

# ─────────────────────────────────────────────────────────────────────────────
# DATA LOADING
# ─────────────────────────────────────────────────────────────────────────────
def parse_filename(fname: str) -> Tuple[str, str, str]:
    parts = Path(fname).stem.split("_")
    return parts[0], parts[1], parts[2]

def scan_dataset(data_root: str, snr_levels: Tuple[str, ...]) -> List[Dict]:
    records = []
    root    = Path(data_root)
    for uav_dir in sorted(root.iterdir()):
        if not uav_dir.is_dir() or not uav_dir.name.startswith("uav"):
            continue
        for snr in snr_levels:
            snr_dir = uav_dir / snr
            if not snr_dir.exists():
                continue
            for img_path in sorted(snr_dir.glob("*.png")):
                try:
                    uav_id, distance, burst_id = parse_filename(img_path.name)
                    if uav_id not in DEVICE_MAP or distance not in DISTANCE_MAP:
                        continue
                    records.append({
                        "path"          : str(img_path),
                        "device_label"  : DEVICE_MAP[uav_id],
                        "distance_label": DISTANCE_MAP[distance],
                        "snr"           : snr,
                        "burst_key"     : (uav_id, distance, burst_id),
                    })
                except Exception:
                    continue
    return records

def burst_level_split(records, val_fraction=0.2, random_state=42):
    burst_to_device = {r["burst_key"]: r["device_label"] for r in records}
    burst_keys      = sorted(burst_to_device.keys())
    burst_labels    = [burst_to_device[bk] for bk in burst_keys]
    train_keys, val_keys = train_test_split(
        burst_keys, test_size=val_fraction,
        stratify=burst_labels, random_state=random_state,
    )
    train_set = set(map(tuple, train_keys))
    val_set   = set(map(tuple, val_keys))
    return (
        [r for r in records if tuple(r["burst_key"]) in train_set],
        [r for r in records if tuple(r["burst_key"]) in val_set],
    )

# ── Copy to RAM-backed tmpfs ──────────────────────────────────────────────────
DST_PATH = cfg.data_root
_src_candidates = [
    "/kaggle/input/datasets/sambhavnayak/genesys-spectrogram-dataset",
    "/kaggle/input/genesys-spectrogram-dataset",
]
if not Path(DST_PATH).exists():
    _src = next((c for c in _src_candidates if Path(c).exists()), None)
    if _src is None:
        raise FileNotFoundError(f"Dataset not found in {_src_candidates}")
    print(f"Copying {_src} → {DST_PATH} …")
    shutil.copytree(_src, DST_PATH)
    print("Copy complete.\n")
else:
    print(f"Dataset already in {DST_PATH} — skipping copy.\n")

print("Scanning dataset...")
all_records = scan_dataset(cfg.data_root, cfg.snr_levels)
print(f"Total images : {len(all_records):,}")
snr_counts = Counter(r["snr"] for r in all_records)
for snr, cnt in sorted(snr_counts.items()):
    print(f"  {snr:<12}: {cnt:,}")

train_records, val_records = burst_level_split(all_records)
overlap = {tuple(r["burst_key"]) for r in train_records} & {tuple(r["burst_key"]) for r in val_records}
print(f"\nTrain images : {len(train_records):,}")
print(f"Val images   : {len(val_records):,}")
print(f"Burst overlap (must be 0): {len(overlap)}")
assert len(overlap) == 0, "DATA LEAKAGE DETECTED"

# ─────────────────────────────────────────────────────────────────────────────
# DATASET & TRANSFORMS
# ─────────────────────────────────────────────────────────────────────────────
IMAGE_CACHE: Dict[str, bytes] = {}

class GenesysDataset(Dataset):
    def __init__(self, records, transform=None):
        self.records   = records
        self.transform = transform

    def __len__(self): return len(self.records)

    def _load_image(self, path):
        if path in IMAGE_CACHE:
            return Image.open(io.BytesIO(IMAGE_CACHE[path])).convert("RGB")
        return Image.open(path).convert("RGB")

    def __getitem__(self, idx):
        r   = self.records[idx]
        img = self._load_image(r["path"])
        if self.transform:
            img = self.transform(img)
        labels = {
            "device_id": torch.tensor(r["device_label"],   dtype=torch.long),
            "distance" : torch.tensor(r["distance_label"], dtype=torch.long),
        }
        return img, labels

_MEAN = [0.485, 0.456, 0.406]
_STD  = [0.229, 0.224, 0.225]

def get_train_transforms(cfg):
    return transforms.Compose([
        transforms.Resize((cfg.img_size, cfg.img_size)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomAffine(degrees=0, translate=(0.05, 0.05), scale=(0.95, 1.05), shear=3),
        transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.1),
        transforms.ToTensor(),
        transforms.Normalize(mean=_MEAN, std=_STD),
        transforms.RandomErasing(p=0.25, scale=(0.02, 0.12)),
    ])

def get_val_transforms(cfg):
    return transforms.Compose([
        transforms.Resize((cfg.img_size, cfg.img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=_MEAN, std=_STD),
    ])

def build_dataloaders(cfg, train_records, val_records):
    train_ds = GenesysDataset(train_records, transform=get_train_transforms(cfg))
    val_ds   = GenesysDataset(val_records,   transform=get_val_transforms(cfg))

    device_labels = [r["device_label"] for r in train_records]
    class_counts  = Counter(device_labels)
    weights       = [1.0 / class_counts[lb] for lb in device_labels]
    sampler       = WeightedRandomSampler(weights, len(weights), replacement=True)

    kw = dict(num_workers=cfg.num_workers, pin_memory=True,
               prefetch_factor=cfg.prefetch_factor, persistent_workers=True)
    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size,
                              sampler=sampler, drop_last=True, **kw)
    val_loader   = DataLoader(val_ds,   batch_size=cfg.batch_size,
                              shuffle=False, **kw)
    return train_loader, val_loader

train_loader, val_loader = build_dataloaders(cfg, train_records, val_records)
print(f"\nTrain batches : {len(train_loader)}")
print(f"Val batches   : {len(val_loader)}")
print(f"Est. epoch time: ~{len(train_loader) * 0.65 / 60:.0f} min on 2×T4")

# ─────────────────────────────────────────────────────────────────────────────
# MIXUP
# ─────────────────────────────────────────────────────────────────────────────
def mixup_data(x, y_dict, alpha=0.3):
    lam   = float(np.random.beta(alpha, alpha)) if alpha > 0 else 1.0
    index = torch.randperm(x.size(0), device=x.device)
    return lam * x + (1.0 - lam) * x[index], y_dict, {k: v[index] for k, v in y_dict.items()}, lam

# ─────────────────────────────────────────────────────────────────────────────
# LOSS
# ─────────────────────────────────────────────────────────────────────────────
class HomoscedasticMultiTaskLoss(nn.Module):
    def __init__(self, task_names, label_smoothing=0.1):
        super().__init__()
        self.task_names = task_names
        self.log_vars   = nn.ParameterDict({t: nn.Parameter(torch.zeros(1)) for t in task_names})
        self.criteria   = nn.ModuleDict({t: nn.CrossEntropyLoss(label_smoothing=label_smoothing) for t in task_names})

    def forward(self, logits, labels):
        losses = {}
        total  = torch.tensor(0.0, device=next(iter(logits.values())).device)
        for t in self.task_names:
            if t not in logits or t not in labels: continue
            ce        = self.criteria[t](logits[t], labels[t])
            lv        = torch.clamp(self.log_vars[t], -2.0, 2.0)
            precision = torch.exp(-lv)
            total     = total + 0.5 * precision * ce + 0.5 * lv
            losses[t] = ce.item()
        losses["total"] = total.item()
        return total, losses

# ─────────────────────────────────────────────────────────────────────────────
# HEAD  (2 tasks only: device_id + distance)
# ─────────────────────────────────────────────────────────────────────────────
class MultiTaskHead(nn.Module):
    def __init__(self, in_features, num_devices, num_distances, dropout=0.4):
        super().__init__()
        hidden = 512
        self.fc = nn.Sequential(
            nn.Linear(in_features, hidden),
            nn.BatchNorm1d(hidden),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.head_device   = nn.Linear(hidden, num_devices)
        self.head_distance = nn.Linear(hidden, num_distances)

    def forward(self, features):
        x = self.fc(features)
        return {"device_id": self.head_device(x), "distance": self.head_distance(x)}

# ─────────────────────────────────────────────────────────────────────────────
# MODEL: EfficientNet-B0  (replaces Xception for ~8× speed gain)
# Xception:        21.9 M params, 224×224 → ~80 min/epoch on 2×T4
# EfficientNet-B0:  5.3 M params, 160×160 → ~7  min/epoch on 2×T4
# ─────────────────────────────────────────────────────────────────────────────
def build_backbone(pretrained=True, timeout_sec=120):
    import threading
    result = {"backbone": None, "error": None}
    def _load():
        try:
            # Xception: num_classes=0 returns 2048-d global avg-pooled features
            result["backbone"] = timm.create_model("xception", pretrained=pretrained, num_classes=0)
        except Exception as e:
            result["error"] = e
    t = threading.Thread(target=_load, daemon=True)
    t.start(); t.join(timeout=timeout_sec)
    if t.is_alive() or result["backbone"] is None:
        reason = result["error"] or f"timed out after {timeout_sec}s"
        print(f"  WARNING: pretrained load failed ({reason}) — random init")
        return timm.create_model("xception", pretrained=False, num_classes=0)
    print(f"  Xception pretrained weights loaded.")
    return result["backbone"]


class XceptionMultiTask(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        self.backbone = build_backbone(pretrained=pretrained)
        self.head     = MultiTaskHead(
            in_features=cfg.backbone_out,
            num_devices=cfg.num_devices,
            num_distances=cfg.num_distances,
            dropout=cfg.dropout,
        )

    def forward(self, x):
        return self.head(self.backbone(x))


# Shape check
_m = XceptionMultiTask(pretrained=False)
_o = _m(torch.randn(2, 3, cfg.img_size, cfg.img_size))
print(f"\nXception | params: {sum(p.numel() for p in _m.parameters()):,}")
print(f"  device_id output : {_o['device_id'].shape}")
print(f"  distance  output : {_o['distance'].shape}")
print(f"  (img_size={cfg.img_size}, ~{int(len(train_loader)*0.4)} sec/epoch est. on 2xT4)")
del _m, _o

# ─────────────────────────────────────────────────────────────────────────────
# FREEZE / UNFREEZE / OPTIMIZER
# ─────────────────────────────────────────────────────────────────────────────
def get_core(model):
    return model.module if isinstance(model, nn.DataParallel) else model

def freeze_backbone(model):
    for p in get_core(model).backbone.parameters(): p.requires_grad = False

def unfreeze_backbone(model):
    for p in get_core(model).backbone.parameters(): p.requires_grad = True

def build_optimizer(model, criterion, cfg, phase):
    head_params      = list(get_core(model).head.parameters())
    criterion_params = list(criterion.parameters())
    if phase == 1:
        # Only head + task weights — backbone frozen
        return torch.optim.AdamW(
            head_params + criterion_params,
            lr=cfg.head_lr, weight_decay=cfg.weight_decay,
        )
    # Phase 2: differential LR — backbone gets 10× lower LR than head
    backbone_params = [p for p in get_core(model).backbone.parameters() if p.requires_grad]
    return torch.optim.AdamW([
        {"params": backbone_params,                "lr": cfg.backbone_lr},
        {"params": head_params + criterion_params, "lr": cfg.head_lr},
    ], weight_decay=cfg.weight_decay)


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# TRAIN / VALIDATE
# ─────────────────────────────────────────────────────────────────────────────
def train_one_epoch(model, loader, criterion, optimizer, scaler, cfg, use_mixup=True, scheduler=None):
    """
    Gradient accumulation over cfg.grad_accum_steps (currently 1 = no accum).
    Scheduler.step() is called after every optimizer step (OneCycleLR requires this).
    """
    model.train()
    total_loss    = 0.0
    task_correct  = {t: 0 for t in TASKS}
    total_samples = 0
    optimizer.zero_grad(set_to_none=True)

    for step, (images, labels) in enumerate(loader):
        images = images.to(device, non_blocking=True)
        labels = {k: v.to(device, non_blocking=True) for k, v in labels.items()}

        if use_mixup and cfg.mixup_alpha > 0:
            images, y_a, y_b, lam = mixup_data(images, labels, alpha=cfg.mixup_alpha)
        else:
            y_a, y_b, lam = labels, labels, 1.0

        with autocast(device_type="cuda", dtype=torch.float16):
            logits    = model(images)
            loss_a, _ = criterion(logits, y_a)
            loss_b, _ = criterion(logits, y_b)
            loss      = (lam * loss_a + (1.0 - lam) * loss_b) / cfg.grad_accum_steps

        scaler.scale(loss).backward()

        if (step + 1) % cfg.grad_accum_steps == 0 or (step + 1) == len(loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=cfg.grad_clip_norm)
            scaler.step(optimizer)
            scaler.update()
            if scheduler is not None:
                scheduler.step()   # OneCycleLR steps per batch
            optimizer.zero_grad(set_to_none=True)

        bs             = images.size(0)
        total_loss    += loss.item() * cfg.grad_accum_steps * bs
        total_samples += bs
        for t in TASKS:
            task_correct[t] += (logits[t].argmax(1) == labels[t]).sum().item()

    return total_loss / total_samples, {t: task_correct[t] / total_samples * 100.0 for t in TASKS}


@torch.no_grad()
def validate(model, loader, criterion):
    model.eval()
    total_loss    = 0.0
    all_preds     = {t: [] for t in TASKS}
    all_labels    = {t: [] for t in TASKS}
    total_samples = 0
    for images, labels in loader:
        images     = images.to(device, non_blocking=True)
        labels_dev = {k: v.to(device, non_blocking=True) for k, v in labels.items()}
        with autocast(device_type="cuda", dtype=torch.float16):
            logits  = model(images)
            loss, _ = criterion(logits, labels_dev)
        bs             = images.size(0)
        total_loss    += loss.item() * bs
        total_samples += bs
        for t in TASKS:
            all_preds[t].extend(logits[t].argmax(1).cpu().tolist())
            all_labels[t].extend(labels[t].tolist())
    metrics = {}
    for t in TASKS:
        yt, yp = np.array(all_labels[t]), np.array(all_preds[t])
        metrics[t] = {
            "accuracy": accuracy_score(yt, yp) * 100.0,
            "f1_score": f1_score(yt, yp, average="weighted", zero_division=0) * 100.0,
        }
    metrics["average"] = {"accuracy": np.mean([metrics[t]["accuracy"] for t in TASKS])}
    return total_loss / total_samples, metrics

# ─────────────────────────────────────────────────────────────────────────────
# TRAINING LOOP
# ─────────────────────────────────────────────────────────────────────────────
print(f"\n{'='*70}")
print(f"Training: XCEPTION (160×160, batch={cfg.batch_size})")
print(f"  Image size  : {cfg.img_size}×{cfg.img_size}")
print(f"  Batch       : {cfg.batch_size}  |  Grad accum: ×{cfg.grad_accum_steps}")
print(f"  Epochs      : {cfg.epochs}  |  Patience: {cfg.patience}")
print(f"  Phase 1 (frozen backbone): epochs 1–{cfg.freeze_epochs}")
print(f"  Phase 2 (full fine-tune) : epochs {cfg.freeze_epochs+1}–{cfg.epochs}")
print(f"  Scheduler   : OneCycleLR (per batch step)")
print(f"{'='*70}")

model = XceptionMultiTask(pretrained=True)
if use_multi_gpu:
    model = nn.DataParallel(model)
    print(f"  DataParallel on {num_gpus} GPUs")
model = model.to(device)

criterion = HomoscedasticMultiTaskLoss(
    task_names=TASKS, label_smoothing=cfg.label_smoothing
).to(device)

# ── Phase 1: frozen backbone, warm up head with OneCycleLR ───────────────────
freeze_backbone(model)
optimizer_p1 = build_optimizer(model, criterion, cfg, phase=1)
# OneCycleLR for phase 1: ramp up + anneal over all freeze_epochs
steps_p1 = cfg.freeze_epochs * len(train_loader)
sched_p1 = torch.optim.lr_scheduler.OneCycleLR(
    optimizer_p1,
    max_lr=cfg.head_lr,
    total_steps=steps_p1,
    pct_start=0.3,
    anneal_strategy="cos",
    div_factor=10,
    final_div_factor=100,
)
scaler = GradScaler(device="cuda")

history      = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": [], "lr": []}
best_val_acc = 0.0
best_metrics = None
patience_counter = 0
phase2_started   = False
optimizer        = optimizer_p1
scheduler        = sched_p1

for epoch in range(cfg.epochs):

    # ── Phase 2 transition ───────────────────────────────────────────────────
    if epoch == cfg.freeze_epochs and not phase2_started:
        print(f"\n  [Epoch {epoch+1}] Unfreezing backbone — starting full fine-tune")
        unfreeze_backbone(model)
        optimizer = build_optimizer(model, criterion, cfg, phase=2)
        # OneCycleLR for phase 2: starts at a lower max_lr for gentle fine-tune
        steps_p2  = (cfg.epochs - cfg.freeze_epochs) * len(train_loader)
        scheduler = torch.optim.lr_scheduler.OneCycleLR(
            optimizer,
            max_lr=[cfg.backbone_lr, cfg.head_lr],  # backbone LR, head LR
            total_steps=steps_p2,
            pct_start=0.1,        # quick warmup
            anneal_strategy="cos",
            div_factor=5,
            final_div_factor=1000,
        )
        phase2_started = True

    train_loss, train_acc = train_one_epoch(
        model, train_loader, criterion, optimizer, scaler, cfg,
        use_mixup=phase2_started, scheduler=scheduler,
    )
    val_loss, val_metrics = validate(model, val_loader, criterion)

    train_avg = np.mean(list(train_acc.values()))
    val_avg   = val_metrics["average"]["accuracy"]

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(train_avg)
    history["val_acc"].append(val_avg)
    history["lr"].append(optimizer.param_groups[-1]["lr"])  # log head LR

    improved = val_avg > best_val_acc
    if improved:
        best_val_acc     = val_avg
        best_metrics     = deepcopy(val_metrics)
        patience_counter = 0
    else:
        patience_counter += 1

    lv_str = " ".join(
        f"{t}:{torch.clamp(criterion.log_vars[t], -2., 2.).item():.3f}"
        for t in TASKS
    )
    tag = " ★" if improved else ""
    print(
        f"Epoch {epoch+1:3d}/{cfg.epochs}"
        f" | LR {optimizer.param_groups[-1]['lr']:.2e}"
        f" | Train Loss {train_loss:.4f}  Acc {train_avg:.2f}%"
        f" | Val Loss {val_loss:.4f}  Acc {val_avg:.2f}%"
        f" | Best {best_val_acc:.2f}%"
        f" | LogVar [{lv_str}]{tag}"
    )
    for t in TASKS:
        print(f"    {t:<14}: Train {train_acc[t]:.2f}%  Val {val_metrics[t]['accuracy']:.2f}%")

    if patience_counter >= cfg.patience:
        print(f"\n  Early stopping at epoch {epoch + 1}")
        break

core        = get_core(model)
final_state = deepcopy(core.state_dict())
print(f"\nTraining complete. Best val acc: {best_val_acc:.2f}%")
for t in TASKS:
    m = best_metrics[t]
    print(f"  {t:<14} Acc {m['accuracy']:.2f}%  F1 {m['f1_score']:.2f}%")

del model, criterion, optimizer, scaler, scheduler
torch.cuda.empty_cache()

# ─────────────────────────────────────────────────────────────────────────────
# EVALUATION UTILITIES
# ─────────────────────────────────────────────────────────────────────────────
TASK_NUM_CLASSES = {"device_id": cfg.num_devices, "distance": cfg.num_distances}

def compute_task_metrics(y_true, y_pred, num_classes):
    acc = accuracy_score(y_true, y_pred) * 100.0
    f1  = f1_score(y_true, y_pred, average="weighted", zero_division=0) * 100.0
    cm  = confusion_matrix(y_true, y_pred, labels=range(num_classes))
    fpr_list, fnr_list = [], []
    for i in range(num_classes):
        tp = cm[i, i]; fn = cm[i, :].sum() - tp
        fp = cm[:, i].sum() - tp; tn = cm.sum() - tp - fn - fp
        fpr_list.append(fp / (fp + tn) if (fp + tn) > 0 else 0.0)
        fnr_list.append(fn / (fn + tp) if (fn + tp) > 0 else 0.0)
    return {"accuracy": acc, "f1": f1, "fpr": np.mean(fpr_list)*100.0, "fnr": np.mean(fnr_list)*100.0}

@torch.no_grad()
def evaluate_records(eval_model, records, cfg):
    if not records: return None
    ds     = GenesysDataset(records, transform=get_val_transforms(cfg))
    loader = DataLoader(ds, batch_size=cfg.batch_size, shuffle=False,
                        num_workers=cfg.num_workers, pin_memory=True,
                        prefetch_factor=cfg.prefetch_factor, persistent_workers=True)
    all_preds  = {t: [] for t in TASKS}
    all_labels = {t: [] for t in TASKS}
    eval_model.eval()
    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        with autocast(device_type="cuda", dtype=torch.float16):
            logits = eval_model(images)
        for t in TASKS:
            all_preds[t].extend(logits[t].argmax(1).cpu().tolist())
            all_labels[t].extend(labels[t].tolist())
    result = {t: compute_task_metrics(np.array(all_labels[t]), np.array(all_preds[t]), TASK_NUM_CLASSES[t]) for t in TASKS}
    result["average"] = {
        "accuracy": np.mean([result[t]["accuracy"] for t in TASKS]),
        "f1"      : np.mean([result[t]["f1"]       for t in TASKS]),
        "fpr"     : np.mean([result[t]["fpr"]       for t in TASKS]),
        "fnr"     : np.mean([result[t]["fnr"]       for t in TASKS]),
    }
    return result

def print_snr_metrics(snr_label, n_samples, result):
    print(f"\n  SNR: {snr_label}  (n={n_samples:,})")
    print(f"  {'Task':<14} {'Accuracy':>10} {'F1':>10} {'FPR':>10} {'FNR':>10}")
    print(f"  {'-'*56}")
    for t in TASKS:
        m = result[t]
        print(f"  {t:<14} {m['accuracy']:>9.2f}% {m['f1']:>9.2f}% {m['fpr']:>9.2f}% {m['fnr']:>9.2f}%")
    a = result["average"]
    print(f"  {'AVERAGE':<14} {a['accuracy']:>9.2f}% {a['f1']:>9.2f}% {a['fpr']:>9.2f}% {a['fnr']:>9.2f}%")

# ─────────────────────────────────────────────────────────────────────────────
# EVALUATION (Per-SNR)
# ─────────────────────────────────────────────────────────────────────────────
eval_model = XceptionMultiTask(pretrained=False)
eval_model.load_state_dict(final_state)
eval_model = eval_model.to(device)
eval_model.eval()

val_by_snr    = {snr: [] for snr in cfg.snr_levels}
for r in val_records:
    val_by_snr[r["snr"]].append(r)

noise_results = {}
print(f"\n{'='*80}")
print(f"PER-SNR EVALUATION: {cfg.model_name.upper()}")
print(f"{'='*80}")
for snr in cfg.snr_levels:
    records_snr = val_by_snr.get(snr, [])
    result      = evaluate_records(eval_model, records_snr, cfg)
    if result is None:
        print(f"\n  SNR: {snr} -- no data, skipping"); continue
    noise_results[snr] = result
    print_snr_metrics(snr, len(records_snr), result)

print(f"\n{'='*80}")
print(f"OVERALL METRICS: {cfg.model_name.upper()}")
print(f"{'='*80}")
overall = evaluate_records(eval_model, val_records, cfg)
print(f"\n  {'Task':<14} {'Accuracy':>10} {'F1':>10} {'FPR':>10} {'FNR':>10}")
print(f"  {'-'*56}")
for t in TASKS:
    m = overall[t]
    print(f"  {t:<14} {m['accuracy']:>9.2f}% {m['f1']:>9.2f}% {m['fpr']:>9.2f}% {m['fnr']:>9.2f}%")
a = overall["average"]
print(f"  {'AVERAGE':<14} {a['accuracy']:>9.2f}% {a['f1']:>9.2f}% {a['fpr']:>9.2f}% {a['fnr']:>9.2f}%")
del eval_model; torch.cuda.empty_cache()

# ─────────────────────────────────────────────────────────────────────────────
# SAVE & ZIP
# ─────────────────────────────────────────────────────────────────────────────
OUTPUT_DIR = f"output_{cfg.model_name}"
os.makedirs(OUTPUT_DIR, exist_ok=True)

torch.save({"model_name": cfg.model_name, "state_dict": final_state,
            "best_val_acc": best_val_acc}, f"{OUTPUT_DIR}/{cfg.model_name}_final.pth")
print(f"\nModel saved : {OUTPUT_DIR}/{cfg.model_name}_final.pth")

with open(f"{OUTPUT_DIR}/history.pkl", "wb") as f:
    pickle.dump(history, f)

results_summary = {
    "timestamp"   : datetime.now().isoformat(),
    "model_name"  : cfg.model_name,
    "best_val_acc": best_val_acc,
    "config": {
        "img_size": cfg.img_size, "batch_size": cfg.batch_size,
        "epochs": cfg.epochs, "patience": cfg.patience,
        "num_devices": cfg.num_devices, "num_distances": cfg.num_distances,
    },
    "per_task_val": {
        t: {"accuracy": best_metrics[t]["accuracy"], "f1_score": best_metrics[t]["f1_score"]}
        for t in TASKS
    },
    "overall_metrics": {t: {k: round(v,2) for k,v in overall[t].items()} for t in list(TASKS)+["average"]},
    "noise_robustness": {
        snr: {t: {k: round(v,2) for k,v in data.items()} for t,data in snr_data.items()}
        for snr, snr_data in noise_results.items()
    },
}
with open(f"{OUTPUT_DIR}/results_summary.json", "w") as f:
    json.dump(results_summary, f, indent=2)
print(f"Results saved: {OUTPUT_DIR}/results_summary.json")

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
epochs_ran = range(1, len(history["val_acc"]) + 1)
ax1.plot(epochs_ran, history["train_loss"], label="Train"); ax1.plot(epochs_ran, history["val_loss"], label="Val")
ax1.set_title("Loss"); ax1.set_xlabel("Epoch"); ax1.legend(); ax1.grid(alpha=0.3)
ax2.plot(epochs_ran, history["train_acc"], label="Train"); ax2.plot(epochs_ran, history["val_acc"], label="Val")
ax2.axhline(best_val_acc, ls="--", c="red", alpha=0.5, label=f"Best {best_val_acc:.1f}%")
ax2.set_title("Avg Accuracy (device_id + distance)"); ax2.set_xlabel("Epoch")
ax2.set_ylim([0, 100]); ax2.legend(); ax2.grid(alpha=0.3)
fig.suptitle(f"{cfg.model_name} @ {cfg.img_size}px | Best val: {best_val_acc:.2f}%")
plt.tight_layout()
plot_path = f"{OUTPUT_DIR}/training_curves.png"
plt.savefig(plot_path, dpi=150, bbox_inches="tight"); plt.close()
print(f"Plot saved  : {plot_path}")

zip_name = f"{cfg.model_name}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
shutil.make_archive(zip_name, "zip", OUTPUT_DIR)
print(f"ZIP created : {zip_name}.zip  |  Contents: {os.listdir(OUTPUT_DIR)}")

try:
    from IPython.display import FileLink, display
    display(FileLink(f"{zip_name}.zip"))
except ImportError:
    print(f"Download: {os.path.abspath(zip_name)}.zip")
